# T2 processing

In [142]:
import pathlib as plib
import logging

import nibabel as nib
import numpy as np
import torch
from scipy.ndimage import gaussian_filter
import os

import plotly.graph_objects as go
import plotly.subplots as psub

logging.basicConfig(level=logging.INFO)

## Setup
Specify path and get Files.
Pymritools is written for torch Tensors, we get the data to tensors in the beginning, keep in mind to get back to numpy in case you use nibabel niftii saving or further processing.

In [143]:
# specify subject ad session
subj = "sub-001"
sess = "ses-04"

path_sub_ses = plib.Path(f"/data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/{subj}/{sess}")
path_t2 = path_sub_ses.joinpath("anat/")
path_afi = path_sub_ses.joinpath("fmap/")
# get mese files
files_t2 = sorted([f for f in path_t2.iterdir() if f.is_file() and ".nii" in f.suffixes and "MESE" in f.stem])
print(f"found t2 files: ")
for f in files_t2:
    print(f"\t\t{f.name}")
# get afi files
files_afi = sorted([f for f in path_afi.iterdir() if f.is_file() and ".nii" in f.suffixes and "stx" in f.stem and "AFI" in f.stem])
print(f"found afi files: ")
for f in files_afi:
    print(f"\t\t{f.name}")
    
# specify tmp or output path if needed
path_tmp = plib.Path(f"/data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/{subj}/{sess}")
path_tmp.mkdir(exist_ok=True, parents=True)
path_tmp_vis = path_tmp.joinpath("figs/")
path_tmp_vis.mkdir(exist_ok=True, parents=True)

found t2 files: 
		sub-001_ses-04_acq-semc_echo-1_MESE.nii
		sub-001_ses-04_acq-semc_echo-2_MESE.nii
		sub-001_ses-04_acq-semc_echo-3_MESE.nii
		sub-001_ses-04_acq-semc_echo-4_MESE.nii
		sub-001_ses-04_acq-semc_echo-5_MESE.nii
		sub-001_ses-04_acq-semc_echo-6_MESE.nii
found afi files: 
		sub-001_ses-04_acq-tr1stx_echo-1_TB1AFI.nii
		sub-001_ses-04_acq-tr2stx_echo-2_TB1AFI.nii


In [144]:
# want them to be in one 4D volume - could also do this prior via e.g. fslmerge -t
# MESE
mese_aff = nib.load(files_t2[0]).affine
mese_data = torch.from_numpy(np.array([nib.load(f).get_fdata() for f in files_t2]))
# want dims to be [nx, ny, nz, ne]
mese_data = torch.movedim(mese_data, 0, -1)
mese_img = nib.Nifti1Image(mese_data.numpy(), mese_aff)
nx, ny, nz, ne = mese_data.shape

# AFI
afi_aff = nib.load(files_afi[0]).affine
afi_data = torch.from_numpy(np.array([nib.load(f).get_fdata() for f in files_afi]))
afi_data = torch.movedim(afi_data, 0, -1)
afi_img = nib.Nifti1Image(afi_data.numpy(), afi_aff)

In [145]:
# at this point we could save the combined volume if wanted
fn_mese4d = path_tmp.joinpath("mese_4d_mag").with_suffix(".nii")
print(f"save file: {fn_mese4d}")
img = nib.Nifti1Image(mese_data.numpy(), affine=mese_aff)
nib.save(img, fn_mese4d)

fn_afi4d = path_tmp.joinpath("afi_4d").with_suffix(".nii")
print(f"save file: {fn_afi4d}")
img = nib.Nifti1Image(afi_data.numpy(), affine=afi_aff)
nib.save(img, fn_afi4d)


save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/mese_4d_mag.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/afi_4d.nii


Usually i resample the afi image straight to the MESE space for later processing (e.g. using FSLeyes and some linear interpolation or BSpline) and i do that with the 4D volume. This probably should find a way (e.g. through ants) to be incorporated straight from python.

In [146]:
# resample the AFI data to the MESE space using a ANTS
print("Resample AFI to MESE space")
fn_afi4d_resampled = path_tmp.joinpath("afi_4d_resampled").with_suffix(".nii")
interpolation_mode = "BSpline" # "Linear" # "NearestNeighbor" # "BSpline" 
os.system(f"/data/u_kuegler_software/git/r2_map_calculation/resample_afi_4D.sh {path_tmp} {fn_afi4d.name} {fn_mese4d.name} {fn_afi4d_resampled.name} {interpolation_mode}")

# reload resampled image
fn = path_tmp.joinpath("afi_4d_resampled").with_suffix(".nii")
print(f"load file: {fn}")
afi_re_img = nib.load(fn)
afi_re_data = torch.from_numpy(afi_re_img.get_fdata())

Resample AFI to MESE space
Cropping reference image to only two echoes
Resampling input image to reference image space
Resampling done
Cleaning up
load file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/afi_4d_resampled.nii


In [147]:
# just a quick look for reference if needed
fig = psub.make_subplots(
    rows=3, cols=ne,
    shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=0.01, vertical_spacing=0.01,
    row_titles=[f"slice: {1 + int((1 + z) / 3 * nz)}" for z in range(2)] + ["afi echoes"],
    column_titles=[f"echo: {e + 1}" for e in range(ne)],
)
for z in range(2):
    for e in range(ne):
        showscale = True if (z == 0 and e == 0) else False
        fig.add_trace(
            go.Heatmap(
                z=mese_data[:, :, int((1 + z) / 3 * nz), e].numpy(), transpose=True,
                zmin=0, zmax=2000, showscale=showscale,
            ),
            row=z + 1, col=e + 1,
        )
    for e in range(afi_data.shape[-1]):
        fig.add_trace(
            go.Heatmap(
                z=afi_re_data[:, :, int(2 / 3 * nz), e].numpy(), transpose=True,
                zmin=0, zmax=2000, showscale=False,
            ),
            row=3, col=e + 1,
        )
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fn = path_tmp_vis.joinpath("mese_4d_mag").with_suffix(".html")
print(f"save file: {fn}")
fig.write_html(fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/figs/mese_4d_mag.html


## Denoise

We want to denoise the data, this includes a 2 step process:
1) Use PCA based signal - noise separation within neighborhood matrices based on small voxel neighborhoods and the echo time series
2) estimate noise distribution and correct for magnitude noise bias

The second is due to us dealing with magnitude data. We get this magnitude data by taking the absolut value of our complex image data across the channels and combining the channels (usually using adaptive combine in Siemens recons, or simple root sum of squares). If initially we would assume gaussian i.i.d noise in the individual channels real and imaginary data, this means we end up with a non-central chi noise distribution in the combined image magnitude data. Thus the noise contributions in each voxel will foll this distribution which is signal dependent. I.e. high SNR voxels have an approximately gaussian noise distribution. Low SNR voxels will have a non-zero offset. In between the distribution kind of shifts between the scenarios depending on the signal value. We want to correct for the offset such that the signal indeed approaches 0 even for low SNR.

There are 2 main caveats. One the noise distribution is not stationary across GRAPPA reconstructed images. One could estimate how the noise changes across the FOV but this would need GRAPPA reconstruction weights, which are not available from the Siemens reco. We could get them using the raw data and reconstructing the data ourselves. This means the noise bias correction will not be perfect but might suffice.
 
Sidenote: if we would use the raw data i would adopt the denoising to a newer version, 1) channel wise denoising in k-space and before recon, this way we would deal with simple gaussian noise, 2) i would adopt the recon moving to J-LORAKS rather than GRAPPA. Both could be really worth it but might be too much work considering we are in the middle of the project.

Hence below, the running version for now.

### Denoising

In [148]:
from pymritools.processing.denoising import denoise_mppca

In [149]:
# assumes torch tensor input
denoised_data, _, _ = denoise_mppca(input_data=mese_data, p=1, device=torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

INFO:pymritools.processing.denoising.mppca.denoise:Set fixed threshold p: 1
Batch denoising: 100%|██████████| 34/34 [00:07<00:00,  4.45it/s]


In [150]:
# we can save this for reference
fn = path_tmp.joinpath("mese_data_denoised").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(denoised_data.numpy(), mese_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/mese_data_denoised.nii


### Noise bias correction

In [151]:
from pymritools.processing.denoising import extract_noise_mask, extract_noise_stats_from_mask, noise_bias_correction

In [152]:
# extract noise voxels from the data ( this should neglect residual grappa recon artifacts )
noise_mask = extract_noise_mask(input_data=mese_data, erode_iter=1)
mese_noise_mean = torch.mean(mese_data[noise_mask])

INFO:pymritools.processing.denoising.mppca.denoise:using autodmri to extract mask
extracting noise voxels, autodmri: 100%|██████████| 3/3 [01:11<00:00, 23.99s/it]


In [153]:
# we can save this for reference
fn = path_tmp.joinpath("mese_noise_mask").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(noise_mask.to(torch.int32).numpy(), mese_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/mese_noise_mask.nii


In [154]:
# just a quick look for reference if needed
fig = psub.make_subplots(
    rows=1, cols=5,
    shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=0.01, vertical_spacing=0.01,
    column_titles=[f"slice: {1 + int((1 + z) / 6 * nz)}" for z in range(5)],
)
for z in range(5):
    showscale = True if (z == 0) else False
    # plot mese data
    fig.add_trace(
        go.Heatmap(
            z=mese_data[:, :, int((1 + z) / 6 * nz), 0].numpy(), transpose=True,
            showscale=showscale, zmin=0, zmax=2000,
        ),
        row=1, col=z + 1,
    )
    # plot identified noise voxels
    indices = torch.nonzero(noise_mask[:, :, int((1 + z) / 6 * nz)])
    fig.add_trace(
        go.Scatter(x=indices[:, 0], y=indices[:, 1], mode="markers", showlegend=False, marker=dict(size=2, color="#3ad673")),
        row=1, col=z + 1,
    )
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fn = path_tmp_vis.joinpath("mese_noise_mask").with_suffix(".html")
print(f"save file: {fn}")
fig.write_html(fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/figs/mese_noise_mask.html


In [155]:
# extract noise stats - give visual path if we want to save the figure
sigma, num_channels = extract_noise_stats_from_mask(input_data=mese_data, mask=noise_mask, path_visuals=path_tmp_vis, )

INFO:root:write file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/figs/noise_histogramm.html


In [156]:
# do noise bias correction on denoised data
denoised_data_nbc = noise_bias_correction(
    denoised_data=denoised_data, sigma=sigma, num_channels=num_channels
)

In [157]:
# we can save this for reference
fn = path_tmp.joinpath("mese_data_denoised_nbc").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(denoised_data_nbc.numpy(), mese_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/mese_data_denoised_nbc.nii


In [158]:
# just a quick look for reference if needed
fig = psub.make_subplots(
    rows=3, cols=ne,
    shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=0.01, vertical_spacing=0.01,
    column_titles=[f"echo: {1 + e}" for e in range(ne)],
    row_titles=["mese data", "denoised mese data", "denoised mese data nbc"]
)
z = nz // 2
for di, d in enumerate([mese_data, denoised_data, denoised_data_nbc]):
    showscale = True if (di == 0) else False
    for e in range(ne):
        # plot data
        # set max intensity quite low on purpose to better spot the differences
        # denoised data (and nbc) should look less granular
        # nbc should have lower values outside brain and in los SNR areas
        fig.add_trace(
            go.Heatmap(
                z=d[:, :, z, e].numpy(), transpose=True,
                showscale=showscale, zmin=0, zmax=1000,
            ),
            row=1 + di, col=1 + e,
        )
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fn = path_tmp_vis.joinpath("mese_denoised_data").with_suffix(".html")
print(f"save file: {fn}")
fig.write_html(fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/figs/mese_denoised_data.html


# B1+ correction
The AFI suffers from error especially for low B1+ regions. Its dynamical range is alright from 80% - 120% B1+, outside there's some bias.
However, due to the way the AFI is calculated from 2 echoes, below the 80% the bias is much bigger than above 120%.
The EMC method below gives an alright estimate for B1+ as well.
Since we have the AFI B1+ estimate i usually combined the 2. Leaning towards the AFI inside its dynamic range and towards the EMC B1+ where the AFI is off.
We can get this information from the AFI echoes and an error propagation through its modelling equation.

Thus we first want to extract the std from AFI individual echoes;

We need some scan parameters here:

| Parameter             | Setting |
|-----------------------|---------|
| Flip Angle [$^\circ$] | 55      |
| n (ratio TR2 / TR1)   | 5       |


In [159]:
from pymritools.modeling.b1_afi import calculate_b1, calculate_error_map
n = 5 # can be extraced from .json files
fa = 55.0

# extract noise voxels from the data again ( this should neglect residual recon artefacts ) 
# use original not resampled data
# a) its quicker
# b) resampling can change noise distribution
noise_mask_afi = extract_noise_mask(input_data=afi_data, erode_iter=0)

INFO:pymritools.processing.denoising.mppca.denoise:using autodmri to extract mask
extracting noise voxels, autodmri: 100%|██████████| 3/3 [00:02<00:00,  1.13it/s]


In [160]:
# we can save this for reference
fn = path_tmp.joinpath("noise_mask_afi").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(noise_mask_afi.to(torch.int32).numpy(), afi_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/noise_mask_afi.nii


In [161]:
b1 = calculate_b1(
    b1_data=afi_data, r_tr21=n, flip_angle_set_deg=fa, smoothing_kernel=3
)
b1_err = calculate_error_map(
    b1_data=afi_data, mask=noise_mask_afi, flip_angle_set_deg=fa, path_visuals=path_tmp
)
# calcualte a relative error map
b1_rel_err = np.divide(
    b1_err.numpy(), b1.numpy(), where=b1.numpy() > 1e-9, out=np.zeros_like(b1.numpy())
) * 100
# catch exploding values
b1_rel_err = np.clip(b1_rel_err, 0, 200)

INFO:root:write file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/noise_histogramm.html
INFO:root:write file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/noise_histogramm.html


In [162]:
print(f"b1 values: {b1.min():.2f}, {b1.max():.2f}")
# we want to make b1 a unitless value, no more percentages anymore
b1_unitless = b1 / 100
print(f"b1 values: {b1_unitless.min():.2f}, {b1_unitless.max():.2f}")

b1 values: 9.57, 132.26
b1 values: 0.10, 1.32


In [163]:
# we can save this for reference
fn_afib1 = path_tmp.joinpath("afi_b1").with_suffix(".nii")
print(f"save file: {fn_afib1}")
i = nib.Nifti1Image(b1_unitless.numpy(), afi_img.affine)
nib.save(i, fn_afib1)
# we can save this for reference
fn_afib1err = path_tmp.joinpath("afi_b1_rel_err").with_suffix(".nii")
print(f"save file: {fn_afib1err}")
i = nib.Nifti1Image(b1_rel_err, afi_img.affine)
nib.save(i, fn_afib1err)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/afi_b1.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/afi_b1_rel_err.nii


In [164]:
# once again we want some resampled version brought into the semc image space, we can use linear interpolation here since the b1 and the rel error maps are assumed to vary smoothly

# resample the AFI_B1 data to the MESE space using a ANTS
print("Resample AFI_B1 to MESE space")
fn_afib1_resampled = path_tmp.joinpath("afi_b1_resampled").with_suffix(".nii")
interpolation_mode = "BSpline" # "Linear" # "NearestNeighbor" # "BSpline" 
os.system(f"/data/u_kuegler_software/git/r2_map_calculation/resample_afi_3D.sh {path_tmp} {fn_afib1.name} {fn_mese4d.name} {fn_afib1_resampled.name} {interpolation_mode}")
print("-------------------------")


# resample the AFI_B1 relative error data to the MESE space using a ANTS
print("Resample AFI_B1_rel_err to MESE space")
fn_afib1err_resampled = path_tmp.joinpath("afi_b1_rel_err_resampled").with_suffix(".nii")
interpolation_mode = "BSpline" # "Linear" # "NearestNeighbor" # "BSpline" 
os.system(f"/data/u_kuegler_software/git/r2_map_calculation/resample_afi_3D.sh {path_tmp} {fn_afib1err.name} {fn_mese4d.name} {fn_afib1err_resampled.name} {interpolation_mode}")
print("-------------------------")

fn = path_tmp.joinpath("afi_b1_resampled").with_suffix(".nii")
print(f"load file: {fn}")
b1_re_img = nib.load(fn)
b1_re = torch.from_numpy(b1_re_img.get_fdata())
fn = path_tmp.joinpath("afi_b1_rel_err_resampled").with_suffix(".nii")
print(f"load file: {fn}")
b1_rel_err_re = torch.from_numpy(nib.load(fn).get_fdata())

Resample AFI_B1 to MESE space
Cropping reference image to only two echoes
Resampling input image to reference image space
Resampling done
Cleaning up
-------------------------
Resample AFI_B1_rel_err to MESE space
Cropping reference image to only two echoes
Resampling input image to reference image space
Resampling done
Cleaning up
-------------------------
load file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/afi_b1_resampled.nii
load file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/afi_b1_rel_err_resampled.nii


# T2 fitting


The Fitting of T2 is based on the EMC module, essentially on creation of a dictionary of signal patterns expected for different tissue T2 and voxel B1+ given the sequence parameters and events.
If the sequence parameters are kept equal for all acquisitions we can use the same lookup dictionary throughout the acquisitions and the fitting is only a pattern matching (brute force vector difference minimization in this case).

I used the sequence parameters of ... to create the lookup table (the code is also in the pymritools module (pymritools.simulation.emc).
To simulate the signal response patterns a simulation of the exact gradient events and pulses using SIEMENS IDEA sequence simulation is necessary. Ideally, we would do this using all baselines and scanning parameters of the different scanning systems used (they could exhibit different gradient amplitudes for example). However, if the sequence parameters are equal we could assume in first approx that the sequence events played out are differing only marginally and use the same dictionary for pattern matching. If we see systematic differences or if we can get the sequences simulated on the respective systems, we can revise this approach.

The main parameters to look out for are the following:

| Parameter                           | Setting                             |
|-------------------------------------|-------------------------------------|
| Flip Angle (Refocusing)  [$^\circ$] | 140                                 |
| ESP / TE                 [ms]       | [9.2, 18.4, 27.6, 36.8, 46.0, 55.2] |
| TR                       [ms]       | 5000                                |
| slice thickness          [mm]       | 0.6                                 |
| bandwidth                [Hz/px]    | 451                                 | 


In [165]:
from pymritools.config.database import DB
from pymritools.modeling.dictionary import r2_pattern_matching
# load the lookup dictionary
path_db = plib.Path("/data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/emc/emc_database_7T_semc_0p6.pkl")
db = DB.load(path_db)
# get torch tensors
db_torch_mag, db_torch_phase = db.get_torch_tensors_t1t2b1e()
# normalize database, use magnitude only for now
db_mag_norm = torch.linalg.norm(db_torch_mag, dim=-1, keepdim=True)
db_torch_mag /= db_mag_norm
# get t2 and b1 values that have been simulated
t1_vals, t2_vals, b1_vals = db.get_t1_t2_b1_values()

# normalize mese data
mese_data_norm = torch.linalg.norm(denoised_data_nbc, dim=-1, keepdim=True)
mese_data_normalized = torch.nan_to_num(
    torch.divide(denoised_data_nbc, mese_data_norm), nan=0.0, posinf=0.0, neginf=0.0
).contiguous()

INFO:pymritools.config.database.db:loading file /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/emc/emc_database_7T_semc_0p6.pkl


We do a poor mans b1 regularization.
First run we extract the b1 map the pattern matching would optimize for without providing the afi input

In [166]:
t2_emc, b1_emc, _ = r2_pattern_matching(
    input_data=mese_data_normalized, db_mag=db_torch_mag, t2_vals=t2_vals, b1_vals=b1_vals, t1_vals=t1_vals, b1_data=None,
    batch_size=2000, device=torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"),
)
b1_emc = torch.from_numpy(gaussian_filter(b1_emc.numpy(), sigma=5))

Batch Processing: 100%|██████████| 1444/1444 [00:19<00:00, 75.43it/s]


In [167]:
# we can save this for reference
fn = path_tmp.joinpath("b1_emc").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(b1_emc.numpy(), mese_img.affine)
nib.save(i, fn)

r2_emc = torch.nan_to_num(torch.divide(torch.ones_like(t2_emc), t2_emc), nan=0.0, posinf=0.0, neginf=0.0)
fn = path_tmp.joinpath("r2_emc").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(r2_emc.numpy(), mese_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/b1_emc.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/r2_emc.nii


Now we want to use the AFI B1 estimate and the calculated afi error maps to calculate a combined B1 regularization map.


In [168]:
# we allow for max 10 % relative error in the afi and do a linear weighting between afi b1 and emc b1, essentially if rel afi error is 0 we trust the afi, if relative error of afi is 10% we trust the emc
regularization_factor = 1 - torch.clip(b1_rel_err_re, 0, 10) / 10
b1_reg = regularization_factor * b1_re + (1 - regularization_factor) * b1_emc
b1_reg = torch.from_numpy(gaussian_filter(b1_reg.numpy(), sigma=2))

this should now have the characteristic central brightening, which the afi captures but emc doesnt always.
also, the afi suffering in low b1 areas is mitigated 

In [169]:
# we can save this for reference
fn = path_tmp.joinpath("b1_reg").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(b1_reg.numpy(), b1_re_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/b1_reg.nii


We now redo the fitting with inputting the b1 regularization. This way the algorithm is not simultaneously optimizing for t2 and b1 but only needs to match the pattern wrt t2

In [170]:
t2, b1_reg_fit, l2_min_err = r2_pattern_matching(
    input_data=mese_data_normalized, db_mag=db_torch_mag, t2_vals=t2_vals, b1_vals=b1_vals, t1_vals=t1_vals, b1_data=b1_reg,
    batch_size=2000, device=torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"),
)

Batch Processing: 100%|██████████| 1444/1444 [01:06<00:00, 21.78it/s]


In [171]:
# want to get r2
div_mask = t2 > 1e-9
r2 = torch.zeros_like(t2)
r2[div_mask] = torch.divide(torch.ones_like(r2[div_mask]), t2[div_mask])
# want to get a rough snr estimate, dividing mese max value with noise mean value
fit_data_reg_snr = torch.max(denoised_data_nbc, dim=-1).values / mese_noise_mean
# make a threshold map for voxel below 5
fit_data_reg_snr_th = (fit_data_reg_snr < 5).to(torch.int32)

In [172]:
# we can save this for reference
fn = path_tmp.joinpath("fit_r2").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(r2.numpy(), mese_img.affine)
nib.save(i, fn)
fn = path_tmp.joinpath("fit_t2").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(t2.numpy(), mese_img.affine)
nib.save(i, fn)
fn = path_tmp.joinpath("fit_b1").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(b1_reg_fit.numpy(), b1_re_img.affine)
nib.save(i, fn)
fn = path_tmp.joinpath("fit_data_reg_snr").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(fit_data_reg_snr.numpy(), mese_img.affine)
nib.save(i, fn)
fn = path_tmp.joinpath("fit_data_reg_snr_thr").with_suffix(".nii")
print(f"save file: {fn}")
i = nib.Nifti1Image(fit_data_reg_snr_th.numpy(), mese_img.affine)
nib.save(i, fn)

save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/fit_r2.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/fit_t2.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/fit_b1.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/fit_data_reg_snr.nii
save file: /data/pt_02262/data/TH_bids/testdata_Taechang/dcm_imported/derivatives/relax_R2/sub-001/ses-04/fit_data_reg_snr_thr.nii


# References
autodmri (for mask extraction): 